In [3]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

df = pd.read_csv('../data/raw/bundesliga_23_24.csv')

df = df[['HomeTeam', 'AwayTeam', 'FTHG', 'FTAG']]

df = df.rename(columns={'FTHG': 'HomeGoals', 'FTAG': 'AwayGoals'})

df.head()

,HomeTeam,AwayTeam,HomeGoals,AwayGoals
0,Werder Bremen,Bayern Munich,0,4
1,Augsburg,M'gladbach,4,4
2,Hoffenheim,Freiburg,1,2
3,Leverkusen,RB Leipzig,3,2
4,Stuttgart,Bochum,5,0


In [ ]:
goal_model_data = pd.concat([
    df[['HomeTeam', 'AwayTeam', 'HomeGoals']].assign(home=1).rename(
        columns={'HomeTeam': 'team', 'AwayTeam': 'opponent', 'HomeGoals': 'goals'}),
    df[['AwayTeam', 'HomeTeam', 'AwayGoals']].assign(home=0).rename(
        columns={'AwayTeam': 'team', 'HomeTeam': 'opponent', 'AwayGoals': 'goals'})
])

print("Data restructured perfectly. Total rows:", len(goal_model_data))
goal_model_data.head()

Data restructured perfectly. Total rows: 612


,team,opponent,goals,home
0,Werder Bremen,Bayern Munich,0,1
1,Augsburg,M'gladbach,4,1
2,Hoffenheim,Freiburg,1,1
3,Leverkusen,RB Leipzig,3,1
4,Stuttgart,Bochum,5,1


In [ ]:
poisson_model = smf.glm(formula="goals ~ home + team + opponent", 
                        data=goal_model_data, 
                        family=sm.families.Poisson()).fit()

poisson_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:                  goals   No. Observations:                  612
Model:                            GLM   Df Residuals:                      576
Model Family:                 Poisson   Df Model:                           35
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -894.19
Date:                Fri, 27 Feb 2026   Deviance:                       579.58
Time:                        05:41:40   Pearson chi2:                     501.
No. Iterations:                     5   Pseudo R-squ. (CS):             0.2683
Covariance Type:            nonrobust                                         
=============================================================================================
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept                     0.3508      0.198      1.774      0.076      -0.037       0.738
team[T.Bayern Munich]         0.6175      0.175      3.521      0.000       0.274       0.961
team[T.Bochum]               -0.1598      0.210     -0.762      0.446      -0.571       0.251
team[T.Darmstadt]            -0.4843      0.231     -2.094      0.036      -0.938      -0.031
team[T.Dortmund]              0.2901      0.187      1.555      0.120      -0.076       0.656
team[T.Ein Frankfurt]         0.0091      0.199      0.046      0.964      -0.382       0.400
team[T.FC Koln]              -0.5813      0.236     -2.460      0.014      -1.045      -0.118
team[T.Freiburg]             -0.1079      0.206     -0.524      0.600      -0.511       0.296
team[T.Heidenheim]           -0.0054      0.200     -0.027      0.978      -0.398       0.387
team[T.Hoffenheim]            0.2854      0.188      1.519      0.129      -0.083       0.654
team[T.Leverkusen]            0.5395      0.177      3.048      0.002       0.193       0.886
team[T.M'gladbach]            0.1214      0.195      0.623      0.533      -0.261       0.504
team[T.Mainz]                -0.2588      0.214     -1.210      0.226      -0.678       0.161
team[T.RB Leipzig]            0.4105      0.182      2.256      0.024       0.054       0.767
team[T.Stuttgart]             0.4235      0.181      2.333      0.020       0.068       0.779
team[T.Union Berlin]         -0.4188      0.225     -1.865      0.062      -0.859       0.021
team[T.Werder Bremen]        -0.0474      0.202     -0.234      0.815      -0.444       0.349
team[T.Wolfsburg]            -0.2034      0.211     -0.964      0.335      -0.617       0.210
opponent[T.Bayern Munich]    -0.2405      0.198     -1.217      0.224      -0.628       0.147
opponent[T.Bochum]            0.2018      0.174      1.159      0.246      -0.139       0.543
opponent[T.Darmstadt]         0.3395      0.168      2.015      0.044       0.009       0.670
opponent[T.Dortmund]         -0.3148      0.200     -1.573      0.116      -0.707       0.077
opponent[T.Ein Frankfurt]    -0.1818      0.192     -0.948      0.343      -0.558       0.194
opponent[T.FC Koln]          -0.0235      0.183     -0.129      0.898      -0.382       0.335
opponent[T.Freiburg]         -0.0394      0.184     -0.214      0.831      -0.401       0.322
opponent[T.Heidenheim]       -0.0873      0.187     -0.467      0.641      -0.454       0.279
opponent[T.Hoffenheim]        0.1133      0.179      0.634      0.526      -0.237       0.464
opponent[T.Leverkusen]       -0.8769      0.242     -3.626      0.000      -1.351      -0.403
opponent[T.M'gladbach]        0.1173      0.178      0.659      0.510      -0.232       0.466
opponent[T.Mainz]            -0.1747      0.191     -0.916      0.360      

In [5]:
# Create a scenario: Bayern at home vs Dortmund
bayern_df = pd.DataFrame({'team': ['Bayern Munich'], 'opponent': ['Dortmund'], 'home': [1]})
bayern_home_xg = poisson_model.predict(bayern_df).iloc[0]

# Create a scenario: Dortmund away vs Bayern
dortmund_df = pd.DataFrame({'team': ['Dortmund'], 'opponent': ['Bayern Munich'], 'home': [0]})
dortmund_away_xg = poisson_model.predict(dortmund_df).iloc[0]

print(f"Pre-Match Expected Goals (xG):")
print(f"Bayern Munich: {bayern_home_xg:.2f}")
print(f"Dortmund: {dortmund_away_xg:.2f}")

Pre-Match Expected Goals (xG):
Bayern Munich: 2.46
Dortmund: 1.49


In [6]:
def simulate_match(home_xg, away_xg, num_simulations=10000):
    # Simulate 10,000 matches in a fraction of a second using the Poisson distribution
    home_goals_simulated = np.random.poisson(home_xg, num_simulations)
    away_goals_simulated = np.random.poisson(away_xg, num_simulations)
    
    # Count the outcomes
    home_wins = np.sum(home_goals_simulated > away_goals_simulated)
    draws = np.sum(home_goals_simulated == away_goals_simulated)
    away_wins = np.sum(home_goals_simulated < away_goals_simulated)
    
    # Calculate percentages
    prob_home = round((home_wins / num_simulations) * 100, 2)
    prob_draw = round((draws / num_simulations) * 100, 2)
    prob_away = round((away_wins / num_simulations) * 100, 2)
    
    return prob_home, prob_draw, prob_away

# Run the simulation using the xG we predicted earlier
home_prob, draw_prob, away_prob = simulate_match(bayern_home_xg, dortmund_away_xg)

print(f"--- 10,000 Match Simulations Complete ---")
print(f"Bayern Munich Win Probability: {home_prob}%")
print(f"Draw Probability:              {draw_prob}%")
print(f"Dortmund Win Probability:      {away_prob}%")

--- 10,000 Match Simulations Complete ---
Bayern Munich Win Probability: 59.07%
Draw Probability:              18.79%
Dortmund Win Probability:      22.14%
